<a href="https://colab.research.google.com/github/ClarenceBallensky/MLP-vs-GCN-ids/blob/main/MLP_vs_GCN_Intrustion_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLP v.s. GCN for Intrusion Detection
#### Clarence Ballensky
Research Question: Can a Graph Convolutional Network (GCN) improve network intrusion detection by leveraging relationships between network flows, compared with a Multilayer Perceptron (MLP), which treats each flow independently?

## Imports

In [6]:
import os, hashlib, urllib.request
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

## Retrieve Data Sample

In [3]:
SAMPLE_URL = "https://github.com/ClarenceBallensky/MLP-vs-GCN-ids/releases/download/v1/ids_sample.parquet"
SAMPLE_PATH = "ids_sample.parquet"
EXPECTED_SHA256 = "f4eb850ee32419da5ab3aee9b33da28b1b3a3ab175c7e7af64a4311163fb03d5"
SEED = 42
np.random.seed(SEED)

if not os.path.exists(SAMPLE_PATH):
    urllib.request.urlretrieve(SAMPLE_URL, SAMPLE_PATH)

with open(SAMPLE_PATH, "rb") as f:
    assert hashlib.sha256(f.read()).hexdigest() == EXPECTED_SHA256, "Dataset file is corrupt or changed"

df = pd.read_parquet(SAMPLE_PATH)
print(df.shape)

(300001, 91)


## Inspect Data Sample
Each section answers a question that affects a later decision:

* Integrity checks: confirm the data won’t break training.
* Time span: decides random vs. time-based splitting.
* Hosts: show whether a few hub IPs will dominate the graph and need a neighbor cap.
* Labels: decide binary vs. multiclass and how to handle the “Attempted” variants.

In [4]:
ID_COLS = ["id", "Flow ID", "Timestamp"]  # unique per row, so ignored when checking duplicates

numeric = df.select_dtypes("number")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
label_counts = df["Label"].value_counts()

# --- Structure and integrity ---
print("shape:", df.shape)
print("\ndtypes:\n", df.dtypes.value_counts())
print("\nNaNs:", df.isna().sum().sum())
print("infs:", np.isinf(numeric).sum().sum())
# Rows identical except for ID columns are kept; they differ in timestamp, so they are distinct events
print("rows identical except ID columns:", df.drop(columns=ID_COLS).duplicated().sum())
print("constant columns:", numeric.columns[numeric.nunique() <= 1].tolist())

# --- Time span (informs the train/test split) ---
print("\ntime span:", df["Timestamp"].min(), "to", df["Timestamp"].max())
print("flows per day:\n", df["Timestamp"].dt.date.value_counts().sort_index())

# --- Hosts (informs the graph edge rule) ---
print("\nunique src IPs:", df["Src IP"].nunique(), "| unique dst IPs:", df["Dst IP"].nunique())
print("busiest dst IPs:\n", df["Dst IP"].value_counts().head(5))
print("busiest src IPs:\n", df["Src IP"].value_counts().head(5))

# --- Labels (informs task framing and metrics) ---
print("\nlabels:")
display(pd.DataFrame({"count": label_counts, "share": (label_counts / len(df)).round(4)}))

# --- Sample rows, transposed so all 91 columns are visible ---
with pd.option_context("display.max_rows", None):
    display(df.head(3).T)

shape: (300001, 91)

dtypes:
 int64             59
float32           27
object             4
datetime64[ns]     1
Name: count, dtype: int64

NaNs: 0
infs: 0
rows identical except ID columns: 9
constant columns: ['Bwd URG Flags']

time span: 2018-02-14 12:28:10.765304 to 2018-03-03 00:26:49.532589
flows per day:
 Timestamp
2018-02-14    27999
2018-02-15    25675
2018-02-16    35093
2018-02-20    28739
2018-02-21    33062
2018-02-22    28827
2018-02-23    28361
2018-02-28    31176
2018-03-01    31108
2018-03-02    29957
2018-03-03        4
Name: count, dtype: int64

unique src IPs: 16212 | unique dst IPs: 7367
busiest dst IPs:
 Dst IP
172.31.0.2         116066
169.254.169.254     13111
172.31.69.25        12044
172.31.69.28         5136
72.21.91.29          1807
Name: count, dtype: int64
busiest src IPs:
 Src IP
18.219.193.20    8517
5.101.40.105     2838
5.101.40.43      2679
212.92.116.6     1575
62.1.169.9       1012
Name: count, dtype: int64

labels:


,count,share
Label,,
BENIGN,281842,0.9395
DoS Hulk,8517,0.0284
DDoS-HOIC,5040,0.0168
FTP-BruteForce - Attempted,1437,0.0048
DDoS-LOIC-HTTP,1410,0.0047
Botnet Ares,680,0.0023
Infiltration - NMAP Portscan,444,0.0015
SSH-BruteForce,442,0.0015
DoS GoldenEye,99,0.0003


,0,1,2
id,5900374,3818247,1176576
Flow ID,172.31.67.71-172.31.0.2-51694-53-17,172.31.69.30-18.219.211.138-61382-8080-6,213.202.230.143-172.31.67.50-2034-3389-6
Src IP,172.31.67.71,172.31.69.30,213.202.230.143
Src Port,51694,61382,2034
Dst IP,172.31.0.2,18.219.211.138,172.31.67.50
Dst Port,53,8080,3389
Protocol,17,6,6
Timestamp,2018-03-02 18:58:42.051908,2018-03-02 17:51:26.827284,2018-03-02 17:05:45.866274
Flow Duration,267,11927,90129254
Total Fwd Packet,1,5,13


### Exploring follow-up questions

In [5]:
attacks = df[df["Label"] != "BENIGN"]

# Are attack types concentrated on specific days? Is each one a few source IPs?
display(pd.crosstab(attacks["Label"], attacks["Timestamp"].dt.date))
print("unique src IPs per attack type:\n", attacks.groupby("Label")["Src IP"].nunique())

# How many flows have same-source neighbors available for graph edges?
flows_per_src = df["Src IP"].map(df["Src IP"].value_counts())
print("\nshare of flows whose src IP appears only once:", (flows_per_src == 1).mean().round(3))
print(flows_per_src.describe())

Timestamp,2018-02-14,2018-02-15,2018-02-16,2018-02-20,2018-02-21,2018-02-22,2018-02-28,2018-03-01,2018-03-02
Label,,,,,,,,,
Botnet Ares,0,0,0,0,0,0,0,0,680
DDoS-HOIC,0,0,0,0,5040,0,0,0,0
DDoS-LOIC-HTTP,0,0,0,1410,0,0,0,0,0
DDoS-LOIC-UDP,0,0,0,11,9,0,0,0,0
DDoS-LOIC-UDP - Attempted,0,0,0,0,1,0,0,0,0
DoS GoldenEye,0,99,0,0,0,0,0,0,0
DoS GoldenEye - Attempted,0,17,0,0,0,0,0,0,0
DoS Hulk,0,0,8517,0,0,0,0,0,0
DoS Slowloris,0,41,0,0,0,0,0,0,0


unique src IPs per attack type:
 Label
Botnet Ares                             10
DDoS-HOIC                               10
DDoS-LOIC-HTTP                          10
DDoS-LOIC-UDP                            9
DDoS-LOIC-UDP - Attempted                1
DoS GoldenEye                            1
DoS GoldenEye - Attempted                1
DoS Hulk                                 1
DoS Slowloris                            1
DoS Slowloris - Attempted                1
FTP-BruteForce - Attempted               2
Infiltration - NMAP Portscan             2
SSH-BruteForce                           1
Web Attack - Brute Force - Attempted     1
Name: Src IP, dtype: int64

share of flows whose src IP appears only once: 0.037
count    300001.000000
mean        719.446039
std        1387.057160
min           1.000000
25%         376.000000
50%         531.000000
75%         600.000000
max        8517.000000
Name: Src IP, dtype: float64


**Findings**
| Observation | Design Choice |
| :--- | :--- |
| Classes are extremely imbalanced; ~94% of the data is labeled `BENIGN` | Scoring must heavily rely on precision, recall, and f1 rather than accuracy alone; model <br> should classify benign vs attack |
| Hub destination IPs are very concentrated (`172.31.0.2` receives ~39% of flows), which is bad for graphing | Use source IPs or exclude hub IPs when graphing |
| Source IPs are shared by many flows (median ~531 flows per source IP), so linking all same-source flows would <br> be far  too dense | Cap each flow's neighbors (nearest k flows in time from the same source IP) |
| One source IP matches an entire attack class: `18.219.193.20` appears in 8,517 flows, which is the <br> same count as DoS Hulk, and DoS Hulk has only that one source IP | This could prove advantageous for the GCN model, so IPs are excluded from the features <br> and the advantage will be noted in the write-up |
| Attack types are concentrated on specific days (e.g. DoS Hulk only on Feb 16) | A stratified random split is preferred to a time-based split |

## Splitting the Data

In [7]:
NON_FEATURE_COLS = ID_COLS + ["Src IP", "Src Port", "Dst IP", "Label", "Attempted Category", "Bwd URG Flags"]
feature_cols = df.columns.drop(NON_FEATURE_COLS)

y = (df["Label"] != "BENIGN").astype(int)  # 1 = attack, 0 = benign

# Row index doubles as the node id when the graph is built, so it must be 0..n-1
assert df.index.equals(pd.RangeIndex(len(df)))

# 70% train / 15% validation / 15% test, stratified on the binary label
train_idx, temp_idx = train_test_split(df.index, test_size=0.30, stratify=y, random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=y[temp_idx], random_state=SEED)

for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(f"{name}: {len(idx):,} flows, {y[idx].mean():.4f} attack share")

train: 210,000 flows, 0.0605 attack share
val: 45,000 flows, 0.0605 attack share
test: 45,001 flows, 0.0605 attack share


## Scaling the Features

In [ ]:
from sklearn.preprocessing import StandardScaler

# Signed log1p tames heavy-tailed counts and rates, and handles any negative placeholder values (e.g. -1)
X = np.sign(df[feature_cols]) * np.log1p(df[feature_cols].abs())

# Fit on training rows only so no validation/test information leaks into the scaling
scaler = StandardScaler().fit(X.loc[train_idx])
X_scaled = scaler.transform(X).astype(np.float32)  # transform every row: the GCN needs features for all nodes

print("feature matrix:", X_scaled.shape, "| all values finite:", np.isfinite(X_scaled).all())